# 11 Classification Evaluation

## Purpose

This notebook runs the final locked test-set evaluation for one saved multi-label glycan subtype classifier.

## Why this notebook matters

Notebook 10 is where model selection happens. That notebook trains the classifier, reviews the validation split, and chooses a threshold. Notebook 11 should stay separate from that tuning loop. Its job is to load the already-selected classifier and threshold, run the held-out test split once, and save the final reporting artifacts.

## Inputs

- one saved classifier `best_model/` folder from notebook 10
- the matching `best_threshold.json`
- the matching `label_vocabulary_snapshot.csv`
- `test_classification.csv` from notebook 09

## Saved outputs in this cleaned version

**Core saved outputs**
- `evaluation_config.json`: records the resolved notebook settings and the saved-output list for this run
- `test_metrics.csv` and `test_metrics.json`: compact overall test-set metrics
- `per_label_metrics.csv`: thresholded precision, recall, F1, and support for every subtype label
- `test_prediction_table.csv`: glycan-level table for manual review

**Supporting saved outputs**
- `roc_auc_per_label.csv` and `average_precision_per_label.csv`: threshold-independent per-label ranking summaries
- `curve_aggregate_summary.csv`: macro and support-weighted summaries for the ROC-AUC and average-precision tables
- `exact_match_summary.csv`: glycan-level summary of whether the full label set was predicted correctly
- `exact_match_roc_curve.png` and `exact_match_pr_curve.png`: saved glycan-level exact-match review figures

**Not saved in this cleaned version**
- the support-weighted weak-label review table, because it is derived from `per_label_metrics.csv` and is mainly a notebook review aid
- label-subset ROC and PR plot files, because you said you do not want a top-supported-label plotting view in this notebook


## User settings

Review this cell before running the notebook. All notebook-specific values that may need editing are collected here so the rest of the notebook can stay focused on the workflow.

**Settings to review**
- `PROJECT_ROOT`: root project folder in Google Drive
- `CLASSIFICATION_PREP_RESULTS_DIRNAME`: notebook-09 results folder to read from
- `TOKENIZER_FAMILY`, `PRETRAIN_EXPERIMENT_NAME`, and `SOURCE_CLASSIFIER_RUN_LABEL`: choose which saved classifier to evaluate
- `EVALUATION_RUN_LABEL`: choose the output folder name for this notebook-11 evaluation run
- `BUILD_EXACT_MATCH_PR_COMPARISON`, `COMPARISON_EVALUATION_RUN_LABEL`, and the comparison labels: optional settings for an overlaid exact-match PR comparison between two notebook-11 runs
- `OVERWRITE_EXISTING_OUTPUTS`: whether this notebook may replace an existing notebook-11 output set
- `MAX_LENGTH` and `EVAL_BATCH_SIZE`: evaluation settings

**Expected output**
- a printed summary of the active notebook settings

**How to interpret the result**
- if the classifier run label or evaluation run label is wrong here, fix it before continuing because later cells will read and write those paths directly
- if you enable the comparison overlay, make sure `COMPARISON_EVALUATION_RUN_LABEL` points to a notebook-11 evaluation folder that already contains `test_prediction_table.csv`


In [ ]:
from pathlib import Path

# Update this path if the project folder lives somewhere else in Google Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# Repository settings used to sync the latest notebook and helper code into Colab.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'

# Choose which notebook-09 results folder should provide the held-out test table.
CLASSIFICATION_PREP_RESULTS_DIRNAME = 'classification_prep'

# Choose the saved classifier run that notebook 11 should evaluate.
TOKENIZER_FAMILY = 'manual'
PRETRAIN_EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2'
SOURCE_CLASSIFIER_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_randominit'

# Save notebook-11 outputs to a dedicated evaluation folder. This lets the
# notebook read from one classifier run while writing a separate test report.
EVALUATION_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_randominit_eval'

# Optional exact-match PR overlay settings. Enable this only after both the
# current evaluation run and the comparison evaluation run already have
# notebook-11 prediction tables available.
BUILD_EXACT_MATCH_PR_COMPARISON = True
COMPARISON_EVALUATION_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_mlm_eval'
PRIMARY_EXACT_MATCH_PR_LABEL = 'Random init'
COMPARISON_EXACT_MATCH_PR_LABEL = 'MLM + classifier'

# Set this to True only when you intentionally want to replace an existing
# notebook-11 evaluation folder for the same evaluation run label.
OVERWRITE_EXISTING_OUTPUTS = False

# Evaluation settings.
MAX_LENGTH = 130
EVAL_BATCH_SIZE = 16

print(f'Project root: {PROJECT_ROOT}')
print(f'GitHub repository: {GITHUB_OWNER}/{REPO_NAME} @ {GITHUB_REF}')
print(f'Classification prep results folder: {CLASSIFICATION_PREP_RESULTS_DIRNAME}')
print(f'Tokenizer family: {TOKENIZER_FAMILY}')
print(f'Pretrain experiment name: {PRETRAIN_EXPERIMENT_NAME}')
print(f'Source classifier run label: {SOURCE_CLASSIFIER_RUN_LABEL}')
print(f'Evaluation run label: {EVALUATION_RUN_LABEL}')
print(f'Build exact-match PR comparison: {BUILD_EXACT_MATCH_PR_COMPARISON}')
print(f'Comparison evaluation run label: {COMPARISON_EVALUATION_RUN_LABEL}')
print(f'Primary PR label: {PRIMARY_EXACT_MATCH_PR_LABEL}')
print(f'Comparison PR label: {COMPARISON_EXACT_MATCH_PR_LABEL}')
print(f'Overwrite existing outputs: {OVERWRITE_EXISTING_OUTPUTS}')
print(f'Max length: {MAX_LENGTH}')
print(f'Evaluation batch size: {EVAL_BATCH_SIZE}')


## Runtime setup

This cell prepares the Colab runtime so the notebook can read artifacts from Google Drive and import the latest project code from GitHub.

**What this cell does**
- mounts Google Drive
- clones the repository if needed
- fast-forward pulls the selected branch
- adds the repository root to the Python import path

**Expected output**
- confirmation that Drive is mounted
- confirmation that the repository was found or cloned
- confirmation of the active repository directory

**How to interpret the result**
- if repository sync fails here, later imports from `src` will fail or use stale code
- if the repository directory is unexpected, the notebook may not be importing the intended project version


In [ ]:
# Standard library imports are needed here because the shared src helpers are
# not available until after the repository has been cloned or updated.
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read prepared classification tables,
# saved thresholds, and classifier checkpoints.
drive.mount('/content/drive')

# Clone the repository in a fresh Colab session, or update the existing clone
# so this notebook uses the latest helper code.
REPO_DIR = Path('/content') / REPO_NAME
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'

if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

print(f"Updating repository to the latest '{GITHUB_REF}' changes...")
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_REF],
    check=True,
)

# Add the repository root to the import path so later cells can load shared
# helpers from the src package.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

print(f'Repository directory: {REPO_DIR}')


## Import shared helpers

This cell imports the libraries and shared helper functions used throughout the notebook.

**What this cell does**
- imports analysis libraries such as `pandas`
- imports the shared notebook-11 helper functions from `src/`

**Expected output**
- no printed output if imports succeed

**How to interpret the result**
- an import error usually means the repository sync failed or the active branch is missing expected helper code


In [ ]:
import pandas as pd

from IPython.display import display

from src.classification_evaluation import (
    build_curve_aggregate_summary,
    build_exact_match_curve_summary,
    build_multilabel_pr_summary,
    build_multilabel_roc_summary,
    build_support_weighted_error_summary,
    build_test_classification_dataset,
    compute_exact_match_confidence_scores,
    compute_exact_match_flags,
    compute_per_label_metrics,
    load_best_threshold,
    load_classifier_artifacts,
    load_exact_match_review_table,
    load_test_classification_table,
    plot_exact_match_monotonic_pr_curve,
    plot_exact_match_monotonic_pr_curve_comparison,
    plot_exact_match_roc_curve,
    prepare_classification_evaluation_run,
    run_classifier_predictions,
    save_classification_evaluation_config,
    save_classification_evaluation_outputs,
)
from src.classification_training import (
    binarize_multilabel_predictions,
    build_classification_prediction_table,
    compute_multilabel_metrics,
)


## Validate the run settings and build the evaluation context

This cell hands the notebook settings to the shared evaluation helper. The helper validates the notebook-09 and notebook-10 input files, builds the standard notebook-11 output folder, and enforces the overwrite policy outside the notebook body.

The same cell also saves an `evaluation_config.json` snapshot so the evaluation folder records exactly which saved classifier and threshold were used.

**Expected output**
- a short path summary for the main inputs and outputs
- confirmation of the resolved evaluation-config path

**How to interpret the result**
- if this cell fails, the problem is usually a wrong project path, a wrong classifier run label, or an overwrite-protection block
- if the printed input or output paths look wrong, stop here and correct the user settings before evaluating the test set


In [ ]:
# Build the validated run context that later cells will reuse for input
# files, output folders, and overwrite protection.
run_context = prepare_classification_evaluation_run(
    project_root=PROJECT_ROOT,
    classification_prep_dirname=CLASSIFICATION_PREP_RESULTS_DIRNAME,
    tokenizer_family=TOKENIZER_FAMILY,
    pretrain_experiment_name=PRETRAIN_EXPERIMENT_NAME,
    source_classifier_run_label=SOURCE_CLASSIFIER_RUN_LABEL,
    evaluation_run_label=EVALUATION_RUN_LABEL,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
    max_length=MAX_LENGTH,
    eval_batch_size=EVAL_BATCH_SIZE,
)

input_paths = run_context['input_paths']
output_paths = run_context['output_paths']
evaluation_settings = run_context['settings']

# Save the resolved notebook settings now so the evaluation folder records
# exactly which classifier artifacts and notebook options were used.
save_classification_evaluation_config(
    evaluation_settings=evaluation_settings,
    output_path=output_paths['evaluation_config_path'],
)

path_summary = pd.DataFrame(
    {
        'item': [
            'classifier_best_model_dir',
            'best_threshold_path',
            'label_vocabulary_path',
            'test_classification_path',
            'results_dir',
            'evaluation_config_path',
        ],
        'value': [
            str(input_paths['classifier_best_model_dir']),
            str(input_paths['best_threshold_path']),
            str(input_paths['label_vocabulary_path']),
            str(input_paths['test_classification_path']),
            str(output_paths['results_dir']),
            str(output_paths['evaluation_config_path']),
        ],
    }
)

display(path_summary)
print(f"Tokenizer family: {evaluation_settings['tokenizer_family']}")
print(f"Pretrain experiment name: {evaluation_settings['pretrain_experiment_name']}")
print(f"Source classifier run label: {evaluation_settings['source_classifier_run_label']}")
print(f"Evaluation run label: {evaluation_settings['evaluation_run_label']}")
print(f"Evaluation config path: {output_paths['evaluation_config_path']}")


## Load the saved classifier, threshold, and test table

This cell loads the saved notebook-10 classifier artifacts and the prepared notebook-09 test table.

**What this cell does**
- loads the saved classifier model and tokenizer
- loads the saved label vocabulary snapshot
- loads the validation-selected threshold
- loads the prepared test classification table

**Expected output**
- the runtime device
- the chosen threshold
- the test row count and label-vocabulary size

**How to interpret the result**
- the threshold should come from notebook 10, not from the test set
- the test row count should look reasonable for the held-out split you expect to evaluate


In [ ]:
# Load the saved classifier, tokenizer, and label vocabulary snapshot
# from the selected notebook-10 classifier run.
artifact_bundle = load_classifier_artifacts(
    model_dir=input_paths['classifier_best_model_dir'],
    label_vocabulary_path=input_paths['label_vocabulary_path'],
)
model = artifact_bundle['model']
tokenizer = artifact_bundle['tokenizer']
runtime_device = artifact_bundle['runtime_device']
label_vocabulary_df = artifact_bundle['label_vocabulary_df']
label_name_to_id = artifact_bundle['label_name_to_id']

# Load the validation-selected threshold and the prepared held-out test table.
best_threshold_row = load_best_threshold(input_paths['best_threshold_path'])
chosen_threshold = float(best_threshold_row['threshold'])
test_df = load_test_classification_table(input_paths['test_classification_path'])

print(f'Runtime device: {runtime_device}')
print(f'Chosen threshold: {chosen_threshold:.2f}')
print(f'Test rows: {len(test_df):,}')
print(f'Label vocabulary size: {len(label_vocabulary_df):,}')


## Build the tokenized test dataset

This cell converts the prepared test dataframe into tokenized model inputs and multi-hot label vectors.

**What this cell does**
- encodes each glycan's subtype labels using the saved label vocabulary
- tokenizes the glycan sequences with the saved classifier tokenizer
- builds the PyTorch dataset used for batched evaluation

**Expected output**
- the number of tokenized test rows

**How to interpret the result**
- the tokenized test row count should match the prepared test table row count


In [ ]:
# Build the held-out test dataset using the saved label vocabulary and
# the tokenizer that matches the saved classifier.
test_dataset_bundle = build_test_classification_dataset(
    test_df=test_df,
    tokenizer=tokenizer,
    label_name_to_id=label_name_to_id,
    max_length=MAX_LENGTH,
)

encoded_test_df = test_dataset_bundle['test_df']
test_dataset = test_dataset_bundle['test_dataset']

print(f'Test dataset rows: {len(test_dataset):,}')


## Run final test-set predictions

This is the core inference step for notebook 11.

**What this cell does**
- runs the saved classifier across the held-out test split
- converts probabilities into binary predictions using the fixed threshold from notebook 10
- builds glycan-level exact-match flags and confidence scores for later review

**Expected output**
- confirmation that the final test predictions finished successfully

**How to interpret the result**
- at this point the model, label vocabulary, and threshold are already fixed, so no test-set tuning should happen after this cell


In [ ]:
# Run one forward-pass sweep across the held-out test split.
prediction_bundle = run_classifier_predictions(
    model=model,
    evaluation_dataset=test_dataset,
    runtime_device=runtime_device,
    batch_size=EVAL_BATCH_SIZE,
)

test_true_labels = prediction_bundle['true_labels']
test_probabilities = prediction_bundle['probabilities']

# Apply the fixed validation-selected threshold so notebook 11 stays separate
# from model or threshold selection.
test_binary_predictions = binarize_multilabel_predictions(
    test_probabilities,
    threshold=chosen_threshold,
)

# Build glycan-level exact-match review values so later tables can show
# whether the full subtype set was predicted correctly for each glycan.
exact_match_flags = compute_exact_match_flags(
    true_labels=test_true_labels,
    predicted_labels=test_binary_predictions,
)
exact_match_confidence_scores = compute_exact_match_confidence_scores(
    predicted_probabilities=test_probabilities,
    threshold=chosen_threshold,
)

print('Final test-set predictions complete.')


## Compute the overall final metrics

This cell builds the compact overall summaries that are usually the first place to look after a finished evaluation.

**What this cell does**
- computes the overall multi-label test metrics
- records the fixed threshold used for those metrics
- builds a separate glycan-level exact-match summary table

**Expected output**
- a one-row overall metric table
- a one-row exact-match summary table

**How to interpret the result**
- the overall metric table answers how the classifier performed across the whole test set
- the exact-match summary is stricter because a glycan only counts as correct when its full label set is correct


In [ ]:
# Compute the compact overall multi-label metrics for the held-out test set.
test_metrics = compute_multilabel_metrics(
    true_labels=test_true_labels,
    predicted_labels=test_binary_predictions,
    predicted_probabilities=test_probabilities,
)
test_metrics['threshold'] = chosen_threshold

# Build a separate glycan-level summary for exact full-label-set correctness.
exact_match_summary_df = build_exact_match_curve_summary(
    exact_match_flags=exact_match_flags,
    exact_match_confidence_scores=exact_match_confidence_scores,
    threshold=chosen_threshold,
)

display(pd.DataFrame([test_metrics]))
display(exact_match_summary_df)


## Review per-label performance tables

This cell builds the detailed per-label evaluation tables.

**What this cell does**
- computes thresholded per-label precision, recall, F1, and support
- computes per-label ROC-AUC and average precision summaries
- computes macro and support-weighted aggregate curve summaries
- builds a support-weighted weak-label review table for notebook inspection only

**Expected output**
- previews of the saved per-label tables
- a preview of the notebook-only weak-label review table
- the aggregate curve summary table

**How to interpret the result**
- `per_label_metrics_df` is the main saved thresholded label-performance table
- `support_weighted_error_summary_df` is a triage aid for notebook review, but it is not saved because it can be regenerated from `per_label_metrics.csv`
- the ROC-AUC and average-precision tables are helpful when you want a threshold-independent per-label ranking summary


In [ ]:
# Build the saved thresholded per-label metric table.
per_label_metrics_df = compute_per_label_metrics(
    true_labels=test_true_labels,
    predicted_labels=test_binary_predictions,
    label_vocabulary_df=label_vocabulary_df,
)

# Build a notebook-only weak-label review table. This is useful for triage,
# but it is derived from per-label metrics and does not need its own saved file.
support_weighted_error_summary_df = build_support_weighted_error_summary(
    per_label_metrics_df=per_label_metrics_df,
    min_support=1,
)

# Build the saved threshold-independent per-label ranking summaries.
roc_summary_df = build_multilabel_roc_summary(
    true_labels=test_true_labels,
    predicted_probabilities=test_probabilities,
    label_vocabulary_df=label_vocabulary_df,
)
pr_summary_df = build_multilabel_pr_summary(
    true_labels=test_true_labels,
    predicted_probabilities=test_probabilities,
    label_vocabulary_df=label_vocabulary_df,
)
curve_aggregate_summary_df = build_curve_aggregate_summary(
    roc_summary_df=roc_summary_df,
    pr_summary_df=pr_summary_df,
)

display(per_label_metrics_df.head(10))
display(support_weighted_error_summary_df.head(10))
display(roc_summary_df.head(10))
display(pr_summary_df.head(10))
display(curve_aggregate_summary_df)


## Plot and save the exact-match review figures

This cell creates the saved glycan-level exact-match ROC and PR figures.

**What this cell does**
- plots the ROC curve for full-label-set exact-match correctness
- plots the monotonic PR curve for full-label-set exact-match correctness
- saves both figures to the notebook-11 evaluation folder

**Expected output**
- one exact-match ROC plot
- one exact-match PR plot

**How to interpret the result**
- these plots summarize glycan-level strict correctness, not per-label behavior
- a glycan counts as correct only when the entire predicted subtype label set matches the true label set


In [ ]:
# Save the glycan-level exact-match ROC and monotonic PR figures.
plot_exact_match_roc_curve(
    exact_match_flags=exact_match_flags,
    exact_match_confidence_scores=exact_match_confidence_scores,
    save_path=output_paths['exact_match_roc_plot_path'],
)
plot_exact_match_monotonic_pr_curve(
    exact_match_flags=exact_match_flags,
    exact_match_confidence_scores=exact_match_confidence_scores,
    save_path=output_paths['exact_match_pr_plot_path'],
)

print(f"Exact-match ROC plot path: {output_paths['exact_match_roc_plot_path']}")
print(f"Exact-match PR plot path: {output_paths['exact_match_pr_plot_path']}")


## Build the human-readable test prediction table

This cell builds the glycan-level review table that is usually the most useful artifact for manual follow-up.

**What this cell does**
- converts the model outputs into a readable per-glycan prediction table
- adds exact-match review columns for full-label-set correctness
- previews the saved table before it is written to Drive

**Expected output**
- the first few rows of the glycan-level prediction table

**How to interpret the result**
- `is_exact_match = 1` means the full subtype label set was predicted correctly for that glycan
- `exact_match_confidence` is a simple confidence-style review score based on the smallest decision margin across the glycan's labels


In [ ]:
# Build the saved glycan-level prediction table using the encoded test
# dataframe, the predicted probabilities, and the thresholded predictions.
test_prediction_table_df = build_classification_prediction_table(
    source_df=encoded_test_df,
    probabilities=test_probabilities,
    predicted_labels=test_binary_predictions,
    label_vocabulary_df=label_vocabulary_df,
)

# Add exact-match review columns so later sorting and filtering can focus on
# glycans where the full subtype label set was or was not predicted correctly.
test_prediction_table_df['is_exact_match'] = exact_match_flags.astype(int)
test_prediction_table_df['exact_match_confidence'] = exact_match_confidence_scores

display(test_prediction_table_df.head(10))


## Optionally overlay exact-match monotonic PR curves

Use this cell when you want a direct glycan-level strict-correctness comparison between the current notebook-11 run and one other notebook-11 evaluation folder.

**What this cell does**
- keeps the current run as the primary curve
- loads `test_prediction_table.csv` from one comparison notebook-11 evaluation folder
- overlays the two exact-match monotonic PR curves in one figure
- saves the comparison figure into the current notebook-11 evaluation folder

**Expected output**
- either a short message saying the comparison was skipped, or one overlaid exact-match PR plot plus a two-row summary table

**How to interpret the result**
- this is a glycan-level strict-correctness comparison, so a glycan counts as correct only when its full subtype label set matches exactly
- if one curve stays above the other across most recall values, that run is ranking exact matches more cleanly


In [ ]:
# Build one optional exact-match monotonic PR comparison overlay for the
# current evaluation run and one other saved notebook-11 evaluation run.
exact_match_pr_comparison_path = output_paths['results_dir'] / 'exact_match_pr_curve_overlay.png'
comparison_pr_overlay_summary_df = None

if BUILD_EXACT_MATCH_PR_COMPARISON:
    comparison_prediction_table_path = (
        PROJECT_ROOT
        / 'results'
        / 'classification_evaluation'
        / TOKENIZER_FAMILY
        / PRETRAIN_EXPERIMENT_NAME
        / COMPARISON_EVALUATION_RUN_LABEL
        / 'test_prediction_table.csv'
    )

    if exact_match_pr_comparison_path.exists() and not OVERWRITE_EXISTING_OUTPUTS:
        raise FileExistsError(
            'Exact-match PR comparison plot already exists. Set '
            'OVERWRITE_EXISTING_OUTPUTS = True if you intentionally want to replace it.'
        )

    comparison_prediction_table_df = load_exact_match_review_table(
        comparison_prediction_table_path,
    )
    comparison_pr_overlay_summary_df = plot_exact_match_monotonic_pr_curve_comparison(
        primary_exact_match_flags=test_prediction_table_df['is_exact_match'].to_numpy(dtype=int),
        primary_exact_match_confidence_scores=test_prediction_table_df['exact_match_confidence'].to_numpy(dtype=float),
        comparison_exact_match_flags=comparison_prediction_table_df['is_exact_match'].to_numpy(dtype=int),
        comparison_exact_match_confidence_scores=comparison_prediction_table_df['exact_match_confidence'].to_numpy(dtype=float),
        primary_label=PRIMARY_EXACT_MATCH_PR_LABEL,
        comparison_label=COMPARISON_EXACT_MATCH_PR_LABEL,
        save_path=exact_match_pr_comparison_path,
    )

    display(comparison_pr_overlay_summary_df)
    print(f'Exact-match PR comparison plot path: {exact_match_pr_comparison_path}')
else:
    print('Skipped exact-match PR comparison because BUILD_EXACT_MATCH_PR_COMPARISON is False.')


## Save the final evaluation outputs

This cell writes the focused notebook-11 output set to Drive.

**What this cell does**
- saves overall metric tables
- saves detailed per-label tables
- keeps the saved exact-match figures in the evaluation folder
- saves the glycan-level prediction table
- prints the final saved-file paths

**Expected output**
- confirmation that all core notebook-11 outputs were saved
- a printed list of saved file paths

**How to interpret the result**
- this printed list is the final manifest for the evaluation run and should match the cleaned output set described at the top of the notebook


In [ ]:
# Save the focused notebook-11 output set. Review-only notebook tables,
# such as the support-weighted weak-label table, are intentionally not saved.
save_classification_evaluation_outputs(
    test_metrics=test_metrics,
    per_label_metrics_df=per_label_metrics_df,
    roc_summary_df=roc_summary_df,
    pr_summary_df=pr_summary_df,
    curve_aggregate_summary_df=curve_aggregate_summary_df,
    exact_match_summary_df=exact_match_summary_df,
    test_prediction_table_df=test_prediction_table_df,
    output_paths=output_paths,
)

print('Saved final evaluation files:')
for output_name, output_path in output_paths.items():
    if output_name == 'results_dir':
        continue
    print(f'- {output_name}: {output_path}')


## Final note

At this point the notebook should have produced a focused final evaluation folder with:
- one compact overall metrics summary
- one exact-match summary table
- full per-label thresholded and threshold-independent tables
- two saved exact-match review figures
- one glycan-level prediction table for manual review

If you later decide you want a different label-level plotting strategy, that can be added back intentionally instead of defaulting to a top-supported-label view.
